In [1]:
# import libraries

from pyspark.sql import SparkSession, functions as f

from pyspark.sql.functions import (
    explode, desc,  row_number, col, try_divide, year, try_to_date, count, 
    countDistinct
)

from pyspark.sql.window import Window

# import pandas as pd


In [2]:
spark = (
    SparkSession.builder
    .appName("Steam EDA")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 10:40:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(spark.version)

4.2.0


In [4]:
# curl https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# record a file in local wget -O steam_game_output.json https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json

In [5]:
from pathlib import Path
from urllib.request import urlretrieve

url = (
    "https://full-stack-bigdata-datasets.s3.amazonaws.com/"
    "Big_Data/Project_Steam/steam_game_output.json"
)
filepath = Path("steam_game_output.json")

if not filepath.exists():
    urlretrieve(url, filepath)

df = spark.read.json(str(filepath), multiLine=True)

In [6]:
# Number of elements in dataframe
print(f"Entries number : {df.count()}")

26/09/15 10:40:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Entries number : 55691


The dataset is to big for json_normalize method

In [7]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [8]:
df.take(1)

[Row(data=Row(appid=10, categories=['Multi-player', 'Valve Anti-Cheat enabled', 'Online PvP', 'Shared/Split Screen PvP', 'PvP'], ccu=13990, developer='Valve', discount='0', genre='Action', header_image='https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513', initialprice='999', languages='English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean', name='Counter-Strike', negative=5199, owners='10,000,000 .. 20,000,000', platforms=Row(linux=True, mac=True, windows=True), positive=201215, price='999', publisher='Valve', release_date='2000/11/1', required_age='0', short_description="Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.", tags=Row(1980s=266, 1990's=1191, 2.5

In [9]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

We observe 23 different values : data is a only a node, categories, platforms and tags contain nested informations. Let's flatened the dataframe to range datas at same level. 

In [10]:
# from pyspark.sql.functions import explode, desc

'''flat_df = df.select("id","data.*", explode("data.categories").alias("category"))
flat_df.display()'''

'flat_df = df.select("id","data.*", explode("data.categories").alias("category"))\nflat_df.display()'

In [12]:
from pyspark.sql.functions import concat_ws

flat_df = df.select("id","data.*", concat_ws(", ", "data.categories").alias("category"))
print(flat_df.count())

55691


In [14]:
# from pyspark.sql import functions as F

flat_df = flat_df.withColumn(
    "platform",
    f.concat_ws(', ', *[f.when(f.col(f"platforms.{f}"), f.lit(f)) for f in ['linux', 'mac', 'windows']])
)
flat_df.display()

AttributeError: 'str' object has no attribute 'when'

In [15]:
flat_df = flat_df.withColumn("tag", f.to_json(f.col("tags")))
print(flat_df.count())

55691


In [16]:
flat_df = flat_df.drop( "categories", "platforms", "tags")

In [17]:
flat_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string (nullable = true)
 |-- category: string (nullable = false)
 |-- tag: string (nullable = true)



In [18]:
# flat_df.write.csv("df.csv", header=True)

Export a csv with the .write method requires missing write right on working directory in databricks free edition. So, let's use pandas method .tocsv()

In [19]:
# import pandas as pd

# flat_df.toPandas().to_csv("df.csv", index=False)

# Explorary dataset analysis

## 1. Explore dataset

In [20]:
len(flat_df.columns)

22

In [21]:
flat_df.count()

55691

The dataframe has 23 columns and 55691 rows.

Now, let's query on the dataframe. Spark lets us run classic SQL queries on your tables, however, using classic SQL in Spark requires you to load the data in memory before running any query. We will use the .createOrReplaceTempView Spark DataFrame method in order to load the data in memory under a certain table name, we will then be able to run SQL queries on it.

In [22]:
flat_df.createOrReplaceTempView('table') # Creates a temporary view of the spark dataframe table in memory under the name
# my_table, which we can now query!

The .sql method lets you write queries in SQL while benefiting from the distributed computing advantages of Spark.

In [23]:
result = spark.sql("SELECT * FROM table LIMIT 1") # filters elements from my_table where position
# show everything
result.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,000,000 .. 20,...|  201215|  999|    Valve|   2000/11/1|      

We observe 
- ccu nombre de joueurs qui jouaient simulatanément au moment où le dataset a été construit
- discount en pourcentage
- prix stocké en centimes de dollars
- owners indique un intervalle pour évaluer le nombre d'acheteurs
- categories la liste varie selon le jeu. Elles permettent aux acheteurs de classer les jeux et fonctionnent comme les étagères d'une bibliothèque.

some problems :
- unused spaces -> sort columns to verify wether it's a display problem or requests problem 
- different spelling : request problem on string
- empty entries : '' on website and platform -> verify other columns
- unused columns : id and appid are identifyers, we use name and can delete them, website, header_image 
- des colonnes numériques au format string -> convertir price, initialprice et discount au format long

We must verify with steam team before treat these data.
- noncompliant entries : are None, ., CD PROJEKT RED, [2.21] real publishers ?
- asiatic characters. We need information even if the langage changes.

Let's sample and observe values on selected columns.

In [24]:
flat_df.select('developer', 'publisher', 'owners', 'ccu', 'website', 'header_image').sample(fraction=0.0001).distinct().show(truncate=False)

+---------------+---------------+-----------+---+------------------------------------------------+----------------------------------------------------------------------------+
|developer      |publisher      |owners     |ccu|website                                         |header_image                                                                |
+---------------+---------------+-----------+---+------------------------------------------------+----------------------------------------------------------------------------+
|Delirious & Co.|Delirious & Co.|0 .. 20,000|0  |https://melonsim.com                            |https://cdn.akamai.steamstatic.com/steam/apps/437550/header.jpg?t=1644720219|
|SandorHQ       |SandorHQ       |0 .. 20,000|0  |https://sandorhq.com/games/cats-make-you-smarter|https://cdn.akamai.steamstatic.com/steam/apps/720670/header.jpg?t=1508427262|
+---------------+---------------+-----------+---+------------------------------------------------+----------------------

In [25]:
spark.sql("select * from table limit 5").show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|                    short_description|type|             website|            category|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+-------------------------------------+----+-----------------

In [26]:
result = spark.sql("select distinct type from table")
result.show(truncate=False)

+--------+
|type    |
+--------+
|hardware|
|game    |
+--------+



In [27]:
result = spark.sql("select * from table where type = 'hardware'")
result.show(truncate=False)

+------+------+---+---------+--------+-----+----------------------------------------------------------------------------+------------+---------+----------+--------+--------------------+--------+-----+-----------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+-----------------------------------------+---------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id    |appid |ccu|developer|discount|genre|header_image                                                                |initialprice|languages|name      |negative|owners    

Steam link est un outil édité par Anima Locus qui permet d'accéder à un jeu par lien sur le téléphone, une tablette, un autre PC et même d'accéder à un jeu hébergé sur le pc d'un ami. Ce n'est pas un jeu en soi. Il convient de supprimer la colonne type.

In [28]:
result = spark.sql("select distinct name from table where name like'Counter-Strike%'")
result.show(truncate=False)

+--------------------------------+
|name                            |
+--------------------------------+
|Counter-Strike Nexon: Studio    |
|Counter-Strike: Source          |
|Counter-Strike: Condition Zero  |
|Counter-Strike                  |
|Counter-Strike: Global Offensive|
+--------------------------------+



In [29]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = flat_df.filter(col(c) == '').count()
    if count > 0:
        print(f"{c}: {count}")

developer: 127


genre: 161


languages: 11


publisher: 132


release_date: 99


short_description: 37


website: 25217


category: 970


There are 55691 rows. Website misses on one half rows. Other missing values should be converted.


Pourquoi il manque certaines dates de release ?

In [30]:
result = spark.sql("select * from table where release_date = ''")
result.show()

+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------------------+
|     id|  appid|ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|                 tag|
+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------------------+
| 102500| 

Quelles sont les valeurs nulles sur les champs numériques ?

In [31]:
from pyspark.sql.functions import col

for c in ['appid', 'ccu', 'negative', 'positive']:
    count = flat_df.filter(col(c).isNull()).count()
    if count > 0:
        print(f"{c}: {count}")

Il ne manque aucune valeur dans les colonnes numériques.

## 2. Clean data

### a) treatments

Let's clean datas. First, delete unused spaces with trim method.

In [32]:
from pyspark.sql.functions import trim
clean_df = spark.sql("SELECT * FROM table").select([trim(col(c)).alias(c) for c in spark.sql("SELECT * FROM table").columns])
flat_df.createOrReplaceTempView('table')
spark.sql("SELECT * FROM table").show()

+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+-------------------

Then, lowercase all values.

In [33]:
from pyspark.sql.functions import lower, col

columns = [['id', 'appid', 'ccu', 'developer', 'discount', 'genre', 'header_image', 'initialprice', 'languages', 'name', 'negative', 'owners', 'positive', 'price', 'publisher', 'release_date', 'required_age', 'short_description', 'type', 'website', 'category', 'platform', 'tag']]

clean_df = clean_df \
    .select( \
    [lower(col(c)).alias(c) for c in columns[0]] \
    )
clean_df.show(5)

{"ts": "2026-09-15 10:46:54.826", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `type`]. SQLSTATE: 42703", "context": {"file": "line 7 in cell [33]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o329.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `type`]. SQLSTATE: 42703;\n'Project [lower(id#1043) AS id#1134, lower(appid#1044) AS appid#1135, lower(ccu#1045) AS ccu#1136, lower(developer#1046) AS developer#1137, lower(discount#1047) AS discount#1138, lower(genre#1048) AS g

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `type`]. SQLSTATE: 42703;
'Project [lower(id#1043) AS id#1134, lower(appid#1044) AS appid#1135, lower(ccu#1045) AS ccu#1136, lower(developer#1046) AS developer#1137, lower(discount#1047) AS discount#1138, lower(genre#1048) AS genre#1139, lower(header_image#1049) AS header_image#1140, lower(initialprice#1050) AS initialprice#1141, lower(languages#1051) AS languages#1142, lower(name#1052) AS name#1143, lower(negative#1053) AS negative#1144, lower(owners#1054) AS owners#1145, lower(positive#1055) AS positive#1146, lower(price#1056) AS price#1147, lower(publisher#1057) AS publisher#1148, lower(release_date#1058) AS release_date#1149, lower(required_age#1059) AS required_age#1150, lower(short_description#1060) AS short_description#1151, lower(type#1061) AS type#1152, lower(website#106

Then, convert numerical columns price, initialprice and discount from string to long.

In [34]:
from pyspark.sql.functions import col

numeric_string_cols = [ 'price', 'initialprice', 'discount']

for c in numeric_string_cols:
    clean_df = clean_df.withColumn(c, col(c).cast('long'))


Next, deal with latest missing values.

In [35]:
from pyspark.sql.functions import col, when

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string' and field.name != 'tag']
none_cols = {'category', 'release_date', 'short_description'}

clean_df = flat_df

for c in string_cols:
    if c in none_cols:
        clean_df = clean_df.withColumn(c, when(col(c) == '', None).otherwise(col(c)))
    else:
        clean_df = clean_df.withColumn(c, when(col(c) == '', 'Unknown').otherwise(col(c)))

Endly, drop unused columns.

In [36]:
clean_df = clean_df.drop('id', 'type', 'header_image', 'website')

### b) verify clean_df dataframe

In [37]:
clean_df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- tag: string (nullable = true)



In [38]:
# Number of elements in dataframe
print(f"Entries number : {clean_df.count()}")

Entries number : 55691


In [0]:
result = spark.sql("select * from table limit 1")
result.show(truncate=False)

In [39]:
clean_df.select('name', 'release_date', 'developer', 'publisher', 'owners', 'ccu').sample(fraction=0.0001).show(truncate=False)

+--------------------------+------------+------------------------+------------------------+------------------+---+
|name                      |release_date|developer               |publisher               |owners            |ccu|
+--------------------------+------------+------------------------+------------------------+------------------+---+
|Guroopia!                 |2019/09/13  |Josh Sullivan           |Josh Sullivan           |0 .. 20,000       |0  |
|Velocity 2X               |2015/08/19  |FuturLab                |Sierra                  |50,000 .. 100,000 |1  |
|Monster Showdown          |2022/01/26  |Virtual Uppercut Studios|Virtual Uppercut Studios|0 .. 20,000       |0  |
|Steel Rats                |2018/11/7   |Tate Multimedia         |Tate Multimedia         |200,000 .. 500,000|2  |
|Turtle Lu                 |2018/03/22  |Falco Software          |Unknown                 |20,000 .. 50,000  |0  |
|Fightttris VR             |2019/02/27  |TECHHOME                |TECHHOME      

Verify missing values

In [40]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = clean_df.filter(col(c) == '').count()
    print(f"{c}: {count}")

id: 0


developer: 0


discount: 0


genre: 0


header_image: 0


...

short_description: 0


type: 0


website: 0


category: 0


tag: 0


## 3. Analysis at the "macro" level

Which publisher has released the most games on Steam?

In [41]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg(
        count('name').alias('game_nb'),
        countDistinct('release_date').alias('release_nb')
        ) \
    .orderBy(desc('release_nb')) \
    .limit(1)
result.show()

+--------------+-------+----------+
|     publisher|game_nb|release_nb|
+--------------+-------+----------+
|Big Fish Games|    422|       403|
+--------------+-------+----------+



What is Valve's position as publisher ?

In [64]:
# from pyspark.sql.functions import count, desc

# result = clean_df \
#     .select(clean_df['publisher'],'name', 'release_date') \
#     .groupBy('publisher') \
#     .filter(clean_df.publisher.isin(['big fish games','valve']))
# result.show()

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

In [43]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

26/09/15 10:50:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 1

+---------+-------+----------+----+
|publisher|game_nb|release_nb|rank|
+---------+-------+----------+----+
+---------+-------+----------+----+



26/09/15 10:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/15 10:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


How many games on steam ?

In [45]:
result = clean_df \
    .select(clean_df['name']) \
    .distinct() \
    .count() 
print(f"Games number : {result}")

Games number : 55430


How many developers indicated on steam ?

In [46]:
result = clean_df \
    .select(clean_df['developer']) \
    .distinct() \
    .count()
print(f"Developer number : {result}")



Developer number : 34773


Which developer most contribute to games on steam ?

In [47]:
result = clean_df \
    .select(clean_df['developer'],'name') \
    .distinct() \
    .groupBy('developer') \
    .agg(count('name').alias('game_nb')) \
    .orderBy(desc('game_nb')) \
    .limit(5)
result.show(truncate=False)

+------------------------+-------+
|developer               |game_nb|
+------------------------+-------+
|Choice of Games         |140    |
|Unknown                 |127    |
|Creobit                 |122    |
|Laush Dmitriy Sergeevich|108    |
|Sokpop Collective       |98     |
+------------------------+-------+



What are the best rated games?

In [49]:
result = clean_df \
    .select(clean_df['name'],'positive') \
    .groupBy('name') \
    .agg(f.sum('positive').alias('positive_nb')) \
    .orderBy(f.desc('positive_nb')) \
    .limit(10)
result.show()

+--------------------+-----------+
|                name|positive_nb|
+--------------------+-----------+
|Counter-Strike: G...|    5943345|
|              Dota 2|    1534895|
|  Grand Theft Auto V|    1229265|
| PUBG: BATTLEGROUNDS|    1185361|
|            Terraria|    1014711|
|Tom Clancy's Rain...|     942910|
|         Garry's Mod|     861240|
|     Team Fortress 2|     846407|
|                Rust|     732520|
|       Left 4 Dead 2|     643836|
+--------------------+-----------+



In [62]:
from pyspark.sql.functions import col, try_divide

result = flat_df \
    .select(flat_df['name'],'positive', 'negative') \
    .groupBy('name') \
    .agg( \
        f.sum('positive').alias('positive_sum'), \
        f.sum('negative').alias('negative_sum') \
        ) \
    .filter((col('positive_sum') > 0) & (col('negative_sum') > 0)) \
    .withColumn('ratio', col('positive_sum') /  col('negative_sum')) \
    .orderBy(f.desc('positive_sum')) \
    .limit(10)
result.show(truncate=False)

+--------------------------------+------------+------------+------------------+
|name                            |positive_sum|negative_sum|ratio             |
+--------------------------------+------------+------------+------------------+
|Counter-Strike: Global Offensive|5943345     |787093      |7.551007314256384 |
|Dota 2                          |1534895     |317916      |4.82798915436782  |
|Grand Theft Auto V              |1229265     |213379      |5.760946484893077 |
|PUBG: BATTLEGROUNDS             |1185361     |908515      |1.3047236424274777|
|Terraria                        |1014711     |22380       |45.34008042895442 |
|Tom Clancy's Rainbow Six Siege  |942910      |143247      |6.582406612354884 |
|Garry's Mod                     |861240      |29998       |28.709913994266284|
|Team Fortress 2                 |846407      |57423       |14.739860334709089|
|Rust                            |732520      |112160      |6.531027104136947 |
|Left 4 Dead 2                   |643836

In absolute value, Counter-Strike: Global Offensive has more positive advices (65,4M). Yet Terraria has a better ratio : 45,34 positive advices for 1 negative advice against 7,5 onpour Counter-Strike. Then, Terraria is proportianaly more liked.

Nous avons besoin d'un autre élément d'information pour répondre. Quel est le le jeu le plus utilisé au moment où le dataset a été construit ?

Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [56]:
from pyspark.sql.functions import (
    col, year, try_to_date, count, countDistinct
)

result = (flat_df \
    .select("name", "release_date") \
    .distinct() \
    .withColumn( \
        "release_year", \
        year(try_to_date(col("release_date"), "yyyy/M/d")) \
    ) \
    .filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .agg( \
        count("*").alias("release_nb"), \
        countDistinct("name").alias("name_nb") \
    ) \
    .filter((col('release_nb') > 0) & (col('name_nb') > 0)) \
    .withColumn('ratio', col('release_nb') /  col('name_nb')) \
    .orderBy(desc("release_year")) \
)

result.show()

+------------+----------+-------+------------------+
|release_year|release_nb|name_nb|             ratio|
+------------+----------+-------+------------------+
|        2022|      7451|   7444|1.0009403546480387|
|        2021|      8805|   8796|1.0010231923601638|
|        2020|      8287|   8278|1.0010872191350568|
|        2019|      6949|   6945| 1.000575953923686|
|        2018|      7663|   7654|1.0011758557616932|
|        2017|      6006|   6003|1.0004997501249375|
|        2016|      4176|   4175|1.0002395209580839|
|        2015|      2566|   2565|1.0003898635477584|
|        2014|      1550|   1550|               1.0|
|        2013|       469|    469|               1.0|
|        2012|       344|    344|               1.0|
|        2011|       267|    267|               1.0|
|        2010|       281|    281|               1.0|
|        2009|       309|    309|               1.0|
|        2008|       159|    159|               1.0|
|        2007|        98|     98|             

Yes. There are years with more releases. 
- In absolute value, 2014 to 2022. There are more releases after the covid's year in 2021 with 8676. 
- In prortional value, 2015 to 2022. Releases become higher than games since 2015. Yet, releases are higher on 2020, the covid's year with 1.0011 releases for one game.  
We observe that Covid accentuated the trend which returns then at normal pace in 2022 with 7401 and 1.00094.

In [57]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+-------------------

How are the prizes distributed? Are there many games with a discount?

In [58]:
result = clean_df \
    .select(clean_df['publisher'], 'name', 'release_date', 'price', 'initialprice', 'discount') \
    .distinct() \
    .filter(clean_df.discount != 0 ) \
    .orderBy('publisher', 'name', 'release_date') \
    .show(truncate=False)

+-----------------------+-------------------------------------------------------+------------+-----+------------+--------+
|publisher              |name                                                   |release_date|price|initialprice|discount|
+-----------------------+-------------------------------------------------------+------------+-----+------------+--------+
| Alon Zubina           |ZAAM                                                   |2020/08/13  |159  |799         |80      |
| Appnori Inc.          |All-In-One Sports VR                                   |2021/02/25  |1399 |1999        |30      |
| BD Games              |刻印战记：冰冻之剑 RE                                  |2022/09/29  |599  |799         |25      |
| Night Steed Games     |Ultimate Solid                                         |2016/10/26  |179  |599         |70      |
| Trinity Project       |Through Abandoned: The Underground City                |2015/07/22  |49   |99          |51      |
| КиКо                  |

In [59]:
clean_df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- tag: string (nullable = true)



Prices don't change ? There is no discount on leaders publisher.

In [60]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+-------------------

In [61]:
result = clean_df \
    .select('name','release_date', 'publisher', 'developer', 'languages', 'owners', 'required_age', 'category', 'platform', 'tag','positive', 'negative', 'price', 'initialprice', 'discount') \
    .show(truncate=False)

{"ts": "2026-09-15 10:52:28.153", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `ccu`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor61.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o648.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `ccu`]. SQLSTATE: 42703;\n'Project [name#1167, release_date#1171, publisher#1170, developer#1161, languages#1166, owners#1168, required_age#1172, category#1176, 'platform, tag#87,

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `platform` cannot be resolved. Did you mean one of the following? [`category`, `appid`, `name`, `tag`, `ccu`]. SQLSTATE: 42703;
'Project [name#1167, release_date#1171, publisher#1170, developer#1161, languages#1166, owners#1168, required_age#1172, category#1176, 'platform, tag#87, positive#49L, negative#46L, price#1169, initialprice#1165, discount#1162]
+- Project [appid#36L, ccu#38L, developer#1161, discount#1162, genre#1163, initialprice#1165, languages#1166, name#1167, negative#46L, owners#1168, positive#49L, price#1169, publisher#1170, release_date#1171, required_age#1172, short_description#1173, category#1176, tag#87]
   +- Project [id#1160, appid#36L, ccu#38L, developer#1161, discount#1162, genre#1163, header_image#1164, initialprice#1165, languages#1166, name#1167, negative#46L, owners#1168, positive#49L, price#1169, publisher#1170, release_date#1171, required_age#1172, sho

What are the most represented languages?

Are there many games prohibited for children under 16/18?